In [ ]:
import pandas as pd
from google.colab import files
upload=files.upload()

Saving english_french.csv to english_french.csv


In [ ]:
import io
df = pd.read_csv(io.BytesIO(upload['english_french.csv']))

# Clean
df = df.dropna()

print(df.head())

  English      French
0     Go.        Va !
1     Go.     Marche.
2     Go.  En route !
3     Go.     Bouge !
4     Hi.     Salut !


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

X_train = train_df['English']
y_train = train_df['French']

X_val = val_df['English']
y_val = val_df['French']

In [ ]:
# baseline
train_sentence = X_train.iloc[0]

print("Training sentence:", repr(train_sentence))
print("Baseline Translation:", baseline_translate(train_sentence))
print("Expected True French:", y_train.iloc[0])

Training sentence: "Do you think I'm fat?"
Baseline Translation: Pensez-vous que je suis gros ?
Expected True French: Penses-tu que je suis grosse ?


In [ ]:
#seq2seq
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenizers
tok_en = Tokenizer()
tok_fr = Tokenizer()

tok_en.fit_on_texts(X_train)
tok_fr.fit_on_texts(y_train)

# Get vocabulary sizes
en_vocab_size = len(tok_en.word_index) + 1
fr_vocab_size = len(tok_fr.word_index) + 1

# Convert to sequences
X_seq = tok_en.texts_to_sequences(X_train)
y_seq = tok_fr.texts_to_sequences(y_train)

# Pad
X_seq = pad_sequences(X_seq, maxlen=20, padding='post')
y_seq = pad_sequences(y_seq, maxlen=20, padding='post')


In [45]:
#training model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed

model = Sequential([
    Embedding(input_dim=en_vocab_size, output_dim=128),
    LSTM(128, return_sequences=True),
    TimeDistributed(Dense(fr_vocab_size, activation='softmax'))
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
model.fit(X_seq, y_seq, epochs=3, batch_size=32)


Epoch 1/3
5746/5746 ━━━━━━━━━━━━━━━━━━━━ 230s 40ms/step - loss: 2.0199
Epoch 2/3
5746/5746 ━━━━━━━━━━━━━━━━━━━━ 223s 39ms/step - loss: 1.4775
Epoch 3/3
5746/5746 ━━━━━━━━━━━━━━━━━━━━ 223s 39ms/step - loss: 1.2788


In [46]:
#pretrained
!pip install transformers sentencepiece torch -q

from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = MarianTokenizer.from_pretrained(model_name)
mt_model = MarianMTModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [47]:
#translation_function
def translate(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    outputs = mt_model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test
print(translate(X_val.iloc[0]))

Si tu me donnes un livre, je le lirai.


In [48]:
#evaluation
from nltk.translate.bleu_score import sentence_bleu

def compute_bleu(reference, prediction):
    return sentence_bleu([reference.split()], prediction.split())

# Example
ref = y_val.iloc[0]
pred = translate(X_val.iloc[0])

print("BLEU:", compute_bleu(ref, pred))

BLEU: 0.4854917717073234


In [49]:
from nltk.translate.bleu_score import sentence_bleu

def translate_seq2seq(text):
    text_seq = tok_en.texts_to_sequences([text])
    text_padded = pad_sequences(text_seq, maxlen=20, padding='post')

    # Predict the output sequence of token probabilities
    predictions = model.predict(text_padded, verbose=0)
    predicted_token_ids = predictions.argmax(axis=-1)
    decoded_sentence = tok_fr.sequences_to_texts(predicted_token_ids)
    return decoded_sentence[0] if decoded_sentence else ""

def compute_bleu(reference, prediction):
    return sentence_bleu([reference.split()], prediction.split())

# Baseline BLEU
baseline_bleu = compute_bleu(y_val.iloc[0], baseline_translate(X_val.iloc[0]))

# Marian BLEU
marian_bleu = compute_bleu(y_val.iloc[0], translate(X_val.iloc[0]))

#Seq2Seq BLEU
seq2seq_translation = translate_seq2seq(X_val.iloc[0])
seq2seq_bleu = compute_bleu(y_val.iloc[0], seq2seq_translation)

# Print it out
print("Seq2Seq translation sample:", seq2seq_translation)
print("Seq2Seq BLEU:", seq2seq_bleu)
print("Baseline BLEU:", baseline_bleu)
print("Marian BLEU:", marian_bleu)


Seq2Seq translation sample: si tu me moi un livre je lire
Seq2Seq BLEU: 1.258141043412406e-231
Baseline BLEU: 0
Marian BLEU: 0.4854917717073234
